# Tau sandbox for BrightEyes HDF5 files

This notebook is meant to test two lifetime paths on saved BrightEyes-MCS `.h5` acquisitions:

1. the DFD trace decay fit taken from the full histogram shape
2. the live `COLOR_LIFETIME` fit used in acquisition, based on the 3-slice `(d1, d2, d3)` compression

Workflow:
- load one `(rep, z)` plane from each file
- build the per-pixel DFD histogram by summing the selected channels
- estimate a per-pixel or global lifetime from the full DFD trace with the same log-linear decay fit used in the GUI trace widget
- estimate the live `COLOR_LIFETIME` lifetime with the same circular-mean 3-sample fit used in `acquisition_loop_process.py`
- compare maps, histograms, and selected-pixel traces


In [ ]:
from pathlib import Path
import json
import math
import re

import h5py
import matplotlib.pyplot as plt
import numpy as np


In [ ]:
# User parameters
H5_FILENAMES = [
    Path(r"C:/Data/26-03-17_Convallarioa_and_FLIMLABS_calibrated_Yellow_Slide/01_Convallaria_DFD_40MHz-17-03-2026-16-59-41.h5"),
    Path(r"C:/Data/26-03-17_Convallarioa_and_FLIMLABS_calibrated_Yellow_Slide/FLIMLABS_Yellow_slide_2_5ns-17-03-2026-16-18-22.h5   "),
]
REP_INDEX = 0
Z_INDEX = 0
CHANNEL_SELECTION = "sum"   # "sum", int, or list[int]
SHOW_PIXEL_EXAMPLE = (50, 50)
BUILD_TRACE_FIT_MAP = True   # set False if the per-pixel full-trace fit is too slow
DELTA_TAU_NS = 0.0           # same hue shift idea as the GUI COLOR_LIFETIME control
LIVE_L_PERCENTILES = (1.0, 99.5)
LIVE_QUALITY_CLIP = (0.0, 1.0)
SPLIT_VIEW_KEY = "live_color_lifetime"  # "live_color_lifetime", "trace_tau_ns", "live_tau_ns", "tau_diff_ns", "brightness"


In [ ]:
def attrs_to_dict(h5obj):
    return {key: h5obj.attrs[key] for key in h5obj.attrs.keys()}


def read_metadata(h5file):
    meta = {}
    for group_name in (
        "configurationSpadFCSmanager",
        "configurationFPGA",
        "configurationGUI",
        "configurationGUI_beforeStart",
        "rawStreamAcquisition",
    ):
        if group_name in h5file:
            meta[group_name] = attrs_to_dict(h5file[group_name])
    return meta


def estimate_t_cycle_ns(metadata):
    raw_cfg = metadata.get("rawStreamAcquisition", {})
    clk_multiplier = max(int(raw_cfg.get("clk_multiplier", 1) or 1), 1)

    dfd_cycle_mhz = raw_cfg.get("dfd_cycle_mhz")
    if dfd_cycle_mhz is None:
        for group in metadata.values():
            if not isinstance(group, dict):
                continue
            if "dfd_cycle_mhz" in group:
                dfd_cycle_mhz = group["dfd_cycle_mhz"]
                break

    if dfd_cycle_mhz is None:
        for group in metadata.values():
            if not isinstance(group, dict):
                continue
            for value in group.values():
                text = value.decode(errors="ignore") if isinstance(value, bytes) else str(value)
                match = re.search(r"(?P<cycle>\d+)M(?P<bins>\d+)", text, re.IGNORECASE)
                if match:
                    parsed_cycle_mhz = int(match.group("cycle"))
                    parsed_bins = int(match.group("bins"))
                    if 3 < parsed_cycle_mhz < 100 and 3 < parsed_bins < 1000:
                        dfd_cycle_mhz = parsed_cycle_mhz
                        break
            if dfd_cycle_mhz is not None:
                break

    if dfd_cycle_mhz is None:
        gui_cfg = metadata.get("configurationGUI", {})
        dfd_cycle_mhz = raw_cfg.get("clock_base_mhz", gui_cfg.get("clock_base", 40))

    return 1e3 / (float(dfd_cycle_mhz) * clk_multiplier)


def pick_channel_data(data_zyxtc, channel_selection):
    if channel_selection == "sum":
        return data_zyxtc.sum(axis=-1)
    if isinstance(channel_selection, int):
        return data_zyxtc[..., channel_selection]
    if isinstance(channel_selection, (list, tuple, np.ndarray)):
        return data_zyxtc[..., list(channel_selection)].sum(axis=-1)
    raise ValueError(f"Unsupported channel selection: {channel_selection!r}")


def split_histogram_in_three(hist_yxb):
    n_bins = hist_yxb.shape[-1]
    edges = np.linspace(0, n_bins, 4, dtype=int)
    d1 = hist_yxb[..., edges[0]:edges[1]].sum(axis=-1)
    d2 = hist_yxb[..., edges[1]:edges[2]].sum(axis=-1)
    d3 = hist_yxb[..., edges[2]:edges[3]].sum(axis=-1)
    return d1, d2, d3, edges


# Exact live COLOR_LIFETIME math copied from acquisition_loop_process.py.
def circular_mean_from_positions_live(weights, phases):
    weights = np.asarray(weights, dtype=float)
    phases = np.asarray(phases, dtype=float)

    angles = 2.0 * np.pi * phases
    z = np.sum(weights * np.exp(1j * angles), axis=-1)

    wsum = np.sum(weights, axis=-1)
    z = np.where(wsum > 0, z / wsum, np.nan + 1j * np.nan)

    mean_angle = np.mod(np.angle(z), 2.0 * np.pi)
    mean_phase = mean_angle / (2.0 * np.pi)

    resultant = np.abs(z)
    std_angle = np.sqrt(np.maximum(0.0, -2.0 * np.log(np.clip(resultant, 1e-15, 1.0))))
    std_phase = std_angle / (2.0 * np.pi)

    return mean_phase, std_phase, resultant


def flim_parameters_from_3samples_live(d1, d2, d3, t_cycle, sample_idx, peak_idx=None, total_bins=None):
    if total_bins is None or total_bins <= 0:
        raise ValueError("total_bins must be provided and > 0")

    weights = np.stack([d1, d2, d3], axis=-1).astype(float)
    sample_idx = np.asarray(sample_idx, dtype=float)
    sample_phase = np.mod(sample_idx / float(total_bins), 1.0)
    phase, std_circ, resultant = circular_mean_from_positions_live(weights, sample_phase)

    tau = phase
    if peak_idx is not None:
        peak_phase = np.mod(np.asarray(peak_idx, dtype=float) / float(total_bins), 1.0)
        tau = np.mod(phase - peak_phase, 1.0)

    brightness = np.sum(weights, axis=-1)
    quality = std_circ
    valid = brightness > 0
    return tau, brightness, quality, valid, resultant


def flim_map_to_hcl_live(tau, brightness, quality, valid, tau_min, tau_max):
    h = np.zeros_like(tau, dtype=float)
    mask = valid & np.isfinite(tau)
    if tau_max > tau_min:
        h[mask] = (tau[mask] - tau_min) / (tau_max - tau_min)
    h = np.clip(h, 0.0, 1.0)

    c = np.zeros_like(tau, dtype=float)
    c[valid] = np.clip(quality[valid], 0.0, 1.0)

    l = brightness.astype(float)
    l[~valid] = 0.0
    return h, c, l


def _lab_f_inv(t):
    delta = 6.0 / 29.0
    return np.where(
        t > delta,
        t ** 3,
        3.0 * (delta ** 2) * (t - 4.0 / 29.0),
    )


def lch_to_srgb(l, c, h):
    angle = 2.0 * np.pi * h
    a = c * np.cos(angle)
    b = c * np.sin(angle)

    fy = (l + 16.0) / 116.0
    fx = fy + a / 500.0
    fz = fy - b / 200.0

    x = 95.047 * _lab_f_inv(fx) / 100.0
    y = 100.0 * _lab_f_inv(fy) / 100.0
    z = 108.883 * _lab_f_inv(fz) / 100.0

    r_lin = 3.2406 * x - 1.5372 * y - 0.4986 * z
    g_lin = -0.9689 * x + 1.8758 * y + 0.0415 * z
    b_lin = 0.0557 * x - 0.2040 * y + 1.0570 * z

    rgb_lin = np.stack([r_lin, g_lin, b_lin], axis=-1)
    threshold = 0.0031308
    rgb = np.where(
        rgb_lin <= threshold,
        12.92 * rgb_lin,
        1.055 * np.power(np.clip(rgb_lin, 0.0, None), 1.0 / 2.4) - 0.055,
    )
    return np.clip(rgb, 0.0, 1.0)


def normalize_l_channel(brightness, valid, percentiles=(1.0, 99.5)):
    l = np.zeros_like(brightness, dtype=float)
    if np.any(valid):
        lo = np.nanpercentile(brightness[valid], percentiles[0])
        hi = np.nanpercentile(brightness[valid], percentiles[1])
        hi = max(hi, lo + 1e-12)
        l[valid] = np.clip((brightness[valid] - lo) / (hi - lo), 0.0, 1.0) * 100.0
    return l


def render_live_color_lifetime(h, c, l_counts, valid, delta_tau_ns, t_cycle_ns, quality_clip=(0.0, 1.0), l_percentiles=(1.0, 99.5)):
    h_shift = 0.0 if t_cycle_ns <= 0 else (delta_tau_ns % t_cycle_ns) / t_cycle_ns
    h_shifted = np.mod(h + h_shift, 1.0)

    c_clip = np.zeros_like(c, dtype=float)
    c_clip[valid] = np.clip(c[valid], quality_clip[0], quality_clip[1])
    c_clip = (c_clip - quality_clip[0]) / max(quality_clip[1] - quality_clip[0], 1e-12)
    c_clip[~valid] = 0.0

    l_norm = normalize_l_channel(l_counts, valid, percentiles=l_percentiles)
    rgb = lch_to_srgb(l_norm, c_clip * 100.0, h_shifted)
    rgb[~valid] = 0.0
    return rgb, h_shifted, c_clip, l_norm


# Exact DFD trace fit logic copied from MainWindow.estimateDfdDecayLifetimeNs.
def estimate_dfd_decay_lifetime_ns(trace_sum, trace_x_seconds, bin_width_ns, nbins, tcycle_s, start_level=0.90, end_level=0.20):
    trace_sum = np.asarray(trace_sum, dtype=float)
    trace_x_seconds = np.asarray(trace_x_seconds, dtype=float)
    if (
        trace_sum.size < 4
        or trace_x_seconds.size != trace_sum.size
        or not np.isfinite(bin_width_ns)
        or bin_width_ns <= 0
        or nbins <= 0
        or not np.isfinite(tcycle_s)
        or tcycle_s <= 0
        or not np.isfinite(start_level)
        or not np.isfinite(end_level)
        or not (0.0 < end_level < start_level < 1.0)
    ):
        return None, None, None, None, None

    peak_idx = int(np.argmax(trace_sum))
    trace_sum_peak0 = np.roll(trace_sum, -peak_idx)
    trace_x_peak0_seconds = np.roll(
        np.mod(trace_x_seconds - trace_x_seconds[peak_idx], tcycle_s),
        -peak_idx,
    )

    peak_value = float(trace_sum_peak0[0])
    if not np.isfinite(peak_value) or peak_value <= 0:
        return None, None, peak_idx, trace_sum_peak0, trace_x_peak0_seconds

    y_start = start_level * peak_value
    idx_start_candidates = np.flatnonzero(trace_sum_peak0 <= y_start)
    fallback_levels = [end_level, 0.30, 0.40]
    fallback_levels = [level for level in fallback_levels if 0.0 < level < start_level]
    fallback_levels = list(dict.fromkeys(fallback_levels))
    selected_end_level = None
    idx_end_candidates = np.asarray([], dtype=int)
    y_end = None
    for candidate_end_level in fallback_levels:
        y_end_candidate = candidate_end_level * peak_value
        idx_end_candidate = np.flatnonzero(trace_sum_peak0 <= y_end_candidate)
        if idx_start_candidates.size != 0 and idx_end_candidate.size != 0:
            selected_end_level = candidate_end_level
            idx_end_candidates = idx_end_candidate
            y_end = y_end_candidate
            break

    if idx_start_candidates.size == 0 or idx_end_candidates.size == 0:
        return None, None, peak_idx, trace_sum_peak0, trace_x_peak0_seconds

    start_idx = int(idx_start_candidates[0])
    end_idx = int(idx_end_candidates[0])
    if end_idx <= start_idx:
        return None, None, peak_idx, trace_sum_peak0, trace_x_peak0_seconds

    y_section = trace_sum_peak0[start_idx : end_idx + 1]
    x_section_seconds = trace_x_peak0_seconds[start_idx : end_idx + 1]
    positive = y_section > 0
    if np.count_nonzero(positive) < 4:
        return None, None, peak_idx, trace_sum_peak0, trace_x_peak0_seconds

    x_fit_bins = np.arange(start_idx, end_idx + 1, dtype=float)[positive]
    x_fit_seconds = x_section_seconds[positive]
    y_fit_input = y_section[positive]
    slope, intercept = np.polyfit(x_fit_bins, np.log(y_fit_input), 1)
    if not np.isfinite(slope) or slope >= 0:
        return None, None, peak_idx, trace_sum_peak0, trace_x_peak0_seconds

    tau_ns = -float(bin_width_ns) / slope
    if not np.isfinite(tau_ns) or tau_ns <= 0:
        return None, None, peak_idx, trace_sum_peak0, trace_x_peak0_seconds

    y_fit = np.exp(intercept + slope * x_fit_bins)
    x_fit_bins_plot = np.mod(x_fit_bins + int(peak_idx), int(nbins))
    x_fit_seconds_plot = np.mod(x_fit_seconds + int(peak_idx) * bin_width_ns * 1e-9, tcycle_s)

    fit_curve = {
        "fit_len": int(end_idx - start_idx + 1),
        "x_fit_bins": x_fit_bins_plot,
        "x_fit_seconds": x_fit_seconds_plot,
        "y_fit": y_fit,
    }
    return float(tau_ns), fit_curve, peak_idx, trace_sum_peak0, trace_x_peak0_seconds


def build_trace_fit_from_histogram(hist_1d, t_cycle_ns):
    hist_1d = np.asarray(hist_1d, dtype=float)
    nbins = hist_1d.size
    if nbins <= 0:
        return {
            "peak_idx": None,
            "trace_sum_peak0": hist_1d,
            "trace_x_bins": np.asarray([], dtype=float),
            "trace_x_seconds": np.asarray([], dtype=float),
            "trace_x_peak0_seconds": np.asarray([], dtype=float),
            "tau_ns": None,
            "fit_curve": None,
        }

    trace_x_bins = np.arange(nbins, dtype=float)
    tcycle_s = t_cycle_ns * 1e-9
    trace_x_seconds = trace_x_bins / max(nbins, 1) * tcycle_s
    bin_width_ns = t_cycle_ns / max(nbins, 1)
    tau_ns, fit_curve, peak_idx, trace_sum_peak0, trace_x_peak0_seconds = estimate_dfd_decay_lifetime_ns(
        hist_1d,
        trace_x_seconds,
        bin_width_ns,
        nbins,
        tcycle_s,
    )
    return {
        "peak_idx": peak_idx,
        "trace_sum_peak0": trace_sum_peak0,
        "trace_x_bins": trace_x_bins,
        "trace_x_seconds": trace_x_seconds,
        "trace_x_peak0_seconds": trace_x_peak0_seconds,
        "tau_ns": tau_ns,
        "fit_curve": fit_curve,
    }


def build_trace_fit_map(hist_yxb, t_cycle_ns, enabled=True):
    tau_map_ns = np.full(hist_yxb.shape[:2], np.nan, dtype=float)
    valid = np.zeros(hist_yxb.shape[:2], dtype=bool)
    if not enabled:
        return tau_map_ns, valid

    for y in range(hist_yxb.shape[0]):
        for x in range(hist_yxb.shape[1]):
            fit = build_trace_fit_from_histogram(hist_yxb[y, x], t_cycle_ns)
            if fit["tau_ns"] is not None:
                tau_map_ns[y, x] = fit["tau_ns"]
                valid[y, x] = True
    return tau_map_ns, valid


def analyze_h5(filename, rep_index=0, z_index=0, channel_selection="sum", build_trace_fit_map_enabled=True, delta_tau_ns=0.0, live_quality_clip=(0.0, 1.0), live_l_percentiles=(1.0, 99.5)):
    filename = Path(filename)
    if not filename.exists():
        raise FileNotFoundError(filename)

    with h5py.File(filename, "r") as h5file:
        metadata = read_metadata(h5file)
        data = h5file["data"]
        plane_zyxtc = data[rep_index, z_index]
        hist_yxb = pick_channel_data(plane_zyxtc, channel_selection)
        datasets = list(h5file.keys())
        data_shape = tuple(data.shape)

    t_cycle_ns = estimate_t_cycle_ns(metadata)
    d1, d2, d3, three_edges = split_histogram_in_three(hist_yxb)
    part_lengths = np.diff(three_edges).astype(float)
    part_starts = three_edges[:-1].astype(float)
    sample_idx = part_starts + 0.5 * np.maximum(part_lengths - 1.0, 0.0)

    tau_live_tcycle, brightness, quality, valid_live, resultant = flim_parameters_from_3samples_live(
        d1,
        d2,
        d3,
        t_cycle=t_cycle_ns * 1e-9,
        sample_idx=sample_idx,
        peak_idx=None,
        total_bins=hist_yxb.shape[-1],
    )
    tau_live_ns = tau_live_tcycle * t_cycle_ns
    h_live_raw, c_live_raw, l_live_counts = flim_map_to_hcl_live(
        tau_live_tcycle,
        brightness,
        quality,
        valid_live,
        tau_min=0.0,
        tau_max=1.0,
    )
    live_color_lifetime, h_live, c_live, l_live = render_live_color_lifetime(
        h_live_raw,
        c_live_raw,
        l_live_counts,
        valid_live,
        delta_tau_ns=delta_tau_ns,
        t_cycle_ns=t_cycle_ns,
        quality_clip=live_quality_clip,
        l_percentiles=live_l_percentiles,
    )

    global_hist = hist_yxb.sum(axis=(0, 1))
    global_trace_fit = build_trace_fit_from_histogram(global_hist, t_cycle_ns)
    tau_trace_map_ns, valid_trace_map = build_trace_fit_map(
        hist_yxb,
        t_cycle_ns,
        enabled=build_trace_fit_map_enabled,
    )
    tau_diff_ns = tau_trace_map_ns - tau_live_ns

    return {
        "filename": filename,
        "label": filename.stem,
        "datasets": datasets,
        "data_shape": data_shape,
        "metadata": metadata,
        "hist_yxb": hist_yxb,
        "global_hist": global_hist,
        "global_trace_fit": global_trace_fit,
        "T_cycle_ns": t_cycle_ns,
        "d1": d1,
        "d2": d2,
        "d3": d3,
        "three_edges": three_edges,
        "brightness": brightness,
        "quality": quality,
        "resultant": resultant,
        "valid_live": valid_live,
        "tau_live_tcycle": tau_live_tcycle,
        "tau_live_ns": tau_live_ns,
        "h_live": h_live,
        "c_live": c_live,
        "l_live": l_live,
        "live_color_lifetime": live_color_lifetime,
        "tau_trace_map_ns": tau_trace_map_ns,
        "valid_trace_map": valid_trace_map,
        "tau_diff_ns": tau_diff_ns,
    }


def build_split_image(left_image, right_image):
    left = np.asarray(left_image)
    right = np.asarray(right_image)

    min_y = min(left.shape[0], right.shape[0])
    min_x = min(left.shape[1], right.shape[1])
    left = left[:min_y, :min_x]
    right = right[:min_y, :min_x]

    split_x = min_x // 2
    split_image = np.empty_like(left)
    split_image[:, :split_x] = left[:, :split_x]
    split_image[:, split_x:] = right[:, split_x:]
    return split_image, split_x


def shared_limits(left_image, right_image, valid_mask=None):
    left = np.asarray(left_image, dtype=float)
    right = np.asarray(right_image, dtype=float)
    if valid_mask is None:
        values = np.concatenate([left.ravel(), right.ravel()])
    else:
        left_valid, right_valid = valid_mask
        values = np.concatenate([
            left[np.asarray(left_valid, dtype=bool)].ravel(),
            right[np.asarray(right_valid, dtype=bool)].ravel(),
        ])
    values = values[np.isfinite(values)]
    if values.size == 0:
        return 0.0, 1.0
    vmin = float(values.min())
    vmax = float(values.max())
    if vmax <= vmin:
        vmax = vmin + 1e-12
    return vmin, vmax


def print_result_summary(result):
    print(f"=== {result['filename']} ===")
    print("datasets:", result["datasets"])
    print("data shape:", result["data_shape"])
    print("plane shape (y, x, bins):", result["hist_yxb"].shape)
    print("T_cycle [ns]:", result["T_cycle_ns"])
    print("3-part edges:", result["three_edges"].tolist())
    print("live valid pixels:", int(result["valid_live"].sum()), "/", result["valid_live"].size)
    print(
        "live tau range [ns]:",
        float(np.nanmin(result["tau_live_ns"][result["valid_live"]])) if np.any(result["valid_live"]) else math.nan,
        float(np.nanmax(result["tau_live_ns"][result["valid_live"]])) if np.any(result["valid_live"]) else math.nan,
    )
    print(
        "trace-fit valid pixels:",
        int(np.count_nonzero(result["valid_trace_map"])),
        "/",
        result["valid_trace_map"].size,
    )
    print(
        "trace-fit tau range [ns]:",
        float(np.nanmin(result["tau_trace_map_ns"])) if np.any(result["valid_trace_map"]) else math.nan,
        float(np.nanmax(result["tau_trace_map_ns"])) if np.any(result["valid_trace_map"]) else math.nan,
    )
    print("global trace-fit tau [ns]:", result["global_trace_fit"]["tau_ns"])
    print()


In [ ]:
if len(H5_FILENAMES) != 2:
    raise ValueError(f"Expected exactly 2 H5 files, got {len(H5_FILENAMES)}")

results = [
    analyze_h5(
        filename,
        rep_index=REP_INDEX,
        z_index=Z_INDEX,
        channel_selection=CHANNEL_SELECTION,
        build_trace_fit_map_enabled=BUILD_TRACE_FIT_MAP,
        delta_tau_ns=DELTA_TAU_NS,
        live_quality_clip=LIVE_QUALITY_CLIP,
        live_l_percentiles=LIVE_L_PERCENTILES,
    )
    for filename in H5_FILENAMES
]

for result in results:
    print_result_summary(result)


In [ ]:
fig, axes = plt.subplots(len(results), 5, figsize=(20, 4 * len(results)), constrained_layout=True, squeeze=False)

panels = [
    ("brightness", "gray", "Counts", None),
    ("tau_trace_map_ns", "turbo", "DFD trace fit [ns]", "valid_trace_map"),
    ("tau_live_ns", "turbo", "Live COLOR_LIFETIME fit [ns]", "valid_live"),
    ("tau_diff_ns", "coolwarm", "Trace fit - live fit [ns]", "valid_trace_map"),
    ("live_color_lifetime", None, "Live COLOR_LIFETIME", None),
]

for row, result in enumerate(results):
    for col, (key, cmap, title, valid_key) in enumerate(panels):
        ax = axes[row, col]
        if cmap is None:
            ax.imshow(result[key])
        else:
            kwargs = {}
            if valid_key is not None:
                valid_mask = result[valid_key]
                valid_values = np.asarray(result[key], dtype=float)[np.asarray(valid_mask, dtype=bool)]
                if valid_values.size:
                    kwargs["vmin"] = float(np.nanpercentile(valid_values, 1.0))
                    kwargs["vmax"] = float(np.nanpercentile(valid_values, 99.0))
            ax.imshow(result[key], cmap=cmap, **kwargs)
        ax.set_title(f"{result['label']} | {title}")
        ax.set_xticks([])
        ax.set_yticks([])

plt.show()


In [ ]:
split_key_to_title = {
    "live_color_lifetime": "Live COLOR_LIFETIME",
    "trace_tau_ns": "DFD trace fit [ns]",
    "live_tau_ns": "Live fit [ns]",
    "tau_diff_ns": "Trace fit - live fit [ns]",
    "brightness": "Counts",
}

split_key_to_result_key = {
    "live_color_lifetime": "live_color_lifetime",
    "trace_tau_ns": "tau_trace_map_ns",
    "live_tau_ns": "tau_live_ns",
    "tau_diff_ns": "tau_diff_ns",
    "brightness": "brightness",
}

split_key_to_cmap = {
    "live_color_lifetime": None,
    "trace_tau_ns": "turbo",
    "live_tau_ns": "turbo",
    "tau_diff_ns": "coolwarm",
    "brightness": "gray",
}

split_key_to_valid = {
    "live_color_lifetime": None,
    "trace_tau_ns": ("valid_trace_map", "valid_trace_map"),
    "live_tau_ns": ("valid_live", "valid_live"),
    "tau_diff_ns": ("valid_trace_map", "valid_trace_map"),
    "brightness": None,
}

if SPLIT_VIEW_KEY not in split_key_to_title:
    raise ValueError(f"Unsupported SPLIT_VIEW_KEY: {SPLIT_VIEW_KEY!r}")

result_key = split_key_to_result_key[SPLIT_VIEW_KEY]
left_image = results[0][result_key]
right_image = results[1][result_key]
split_image, split_x = build_split_image(left_image, right_image)

fig, ax = plt.subplots(figsize=(9, 9), constrained_layout=True)
cmap = split_key_to_cmap[SPLIT_VIEW_KEY]
if cmap is None:
    ax.imshow(split_image)
else:
    valid_keys = split_key_to_valid[SPLIT_VIEW_KEY]
    valid_mask = None if valid_keys is None else (results[0][valid_keys[0]], results[1][valid_keys[1]])
    vmin, vmax = shared_limits(left_image, right_image, valid_mask=valid_mask)
    im = ax.imshow(split_image, cmap=cmap, vmin=vmin, vmax=vmax)
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
ax.axvline(split_x - 0.5, color="white", lw=2, alpha=0.9)
ax.set_title(
    f"Split {split_key_to_title[SPLIT_VIEW_KEY]}\n"
    f"left: {results[0]['label']} | right: {results[1]['label']}"
)
ax.set_xticks([])
ax.set_yticks([])
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4), constrained_layout=True)

hist_specs = [
    ("tau_trace_map_ns", "DFD trace fit [ns]", "valid_trace_map", None),
    ("tau_live_ns", "Live fit [ns]", "valid_live", None),
    ("tau_diff_ns", "Trace fit - live fit [ns]", "valid_trace_map", None),
]

for ax, (key, title, valid_key, hist_range) in zip(axes, hist_specs):
    for result in results:
        valid = np.asarray(result[valid_key], dtype=bool)
        values = np.asarray(result[key], dtype=float)[valid]
        values = values[np.isfinite(values)]
        if values.size:
            ax.hist(values, bins=100, range=hist_range, alpha=0.45, label=result["label"])
    ax.set_title(title)
    ax.legend()

plt.show()


In [ ]:
fig, axes = plt.subplots(len(results), 2, figsize=(14, 4 * len(results)), constrained_layout=True, squeeze=False)

for row, result in enumerate(results):
    hist_yxb = result["hist_yxb"]
    y0, x0 = SHOW_PIXEL_EXAMPLE
    y0 = min(max(int(y0), 0), hist_yxb.shape[0] - 1)
    x0 = min(max(int(x0), 0), hist_yxb.shape[1] - 1)

    pixel_hist = hist_yxb[y0, x0]
    pixel_trace_fit = build_trace_fit_from_histogram(pixel_hist, result["T_cycle_ns"])
    global_trace_fit = result["global_trace_fit"]

    ax = axes[row, 0]
    ax.plot(result["global_hist"], color="black", lw=1.5, label="global histogram")
    if global_trace_fit["fit_curve"] is not None:
        ax.plot(global_trace_fit["fit_curve"]["x_fit_bins"], global_trace_fit["fit_curve"]["y_fit"], color="cyan", lw=2, label=f"global DFD fit ({global_trace_fit['tau_ns']:.3f} ns)")
    ax.set_title(f"{result['label']} | global DFD trace fit")
    ax.set_xlabel("DFD bin")
    ax.set_ylabel("counts")
    ax.legend()
    ax.set_yscale("log")

    ax = axes[row, 1]
    three_edges = result["three_edges"]
    ax.plot(pixel_hist, color="black", lw=1.5, label="pixel histogram")
    ax.axvspan(three_edges[0], three_edges[1], color="red", alpha=0.15, label="d1")
    ax.axvspan(three_edges[1], three_edges[2], color="green", alpha=0.15, label="d2")
    ax.axvspan(three_edges[2], three_edges[3], color="blue", alpha=0.15, label="d3")
    if pixel_trace_fit["fit_curve"] is not None:
        ax.plot(pixel_trace_fit["fit_curve"]["x_fit_bins"], pixel_trace_fit["fit_curve"]["y_fit"], color="cyan", lw=2, label=f"pixel DFD fit ({pixel_trace_fit['tau_ns']:.3f} ns)")
    ax.set_title(
        f"{result['label']} | Pixel ({y0}, {x0})\n"
        f"trace_tau={pixel_trace_fit['tau_ns'] if pixel_trace_fit['tau_ns'] is not None else float('nan'):.3f} ns, "
        f"live_tau={result['tau_live_ns'][y0, x0]:.3f} ns, "
        f"quality={result['quality'][y0, x0]:.3f}, "
        f"counts={result['brightness'][y0, x0]:.1f}"
    )
    ax.set_xlabel("DFD bin")
    ax.set_ylabel("counts")
    ax.legend()
    ax.set_yscale("log")

plt.show()


In [ ]:
all_metadata = {result["label"]: result["metadata"] for result in results}
all_metadata
